# 7.1 결정 트리: 구조, 지니불순도, 정보이득 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter07_1_decision_tree.ipynb)

책 본문: [7.1 결정 트리: 구조, 지니불순도, 정보이득](https://smhanlab.com/book-ml/kor/ml1/chapter07/1.html)

이 노트북은 책 7.1절의 모든 수치를 코드로 재현합니다:
(1) 6개 예시로 **지니불순도·엔트로피·정보이득**을 손계산과 똑같이 계산,
(2) "잘못 고를 확률" \(G=2p(1-p)\)와 "비트 수" \(H\)의 **곡선**,
(3) 그 예시로 자란 **트리 구조**(graphviz),
(4) 유방암 데이터에서 **트리 한 개를 읽고** `export_text`/`decision_path`/`feature_importances_`,
(5) 깊이를 늘릴 때의 **U자 곡선**(train 1.0으로 가면서 test가 꺾임),
(6) 2D 장난감 데이터에서 **계단형 결정 경계**(깊이 3 vs 완전).
numpy/matplotlib/sklearn/graphviz만 씁니다.

## 0. 설정: 한글 폰트와 import

그래프의 한글 라벨이 깨지지 않도록 CJK 폰트를 골라둡니다 (Colab에 기본 설치).
그림은 `kor/src/images/`에 SVG로 저장합니다 — Colab에서는 `/tmp`로 바꾸면 됩니다.

In [1]:
import math
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.datasets import load_breast_cancer, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
import graphviz

# 한글 폰트 (Colab에 기본 설치)
for _f in ["Noto Sans CJK KR", "NanumGothic", "Malgun Gothic", "AppleGothic"]:
    if any(_f.lower() == x.name.lower() for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.sans-serif"] = [_f, "DejaVu Sans"]
        break
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["svg.fonttype"] = "path"   # SVG 내보낼 때 글자를 path로 변환

IMG = "/home/smhan/book-ml/kor/src/images"   # Colab에서는 "/tmp"
os.makedirs(IMG, exist_ok=True)
print("IMG =", IMG)

IMG = /home/smhan/book-ml/kor/src/images


## 1. 손으로 계산한 지니·엔트로피·정보이득을 코드로 (본문 '손으로 한 번'의 수치)

본문 "손으로 한 번"의 데이터: 특징 \(x=[1,2,3,4,5,6]\), 라벨
\([A,A,A,B,B,B]\). 부모(전체)의 지니/엔트로피와, 두 분기 후보
(\(x\le1\), \(x\le3\))의 **가중평균 불순도**와 **정보이득**을 계산합니다.
본문의 손계산(0.500/1.000, 그리고 0.100/0.500, 0.191/1.000)과 정확히
일치해야 합니다.

In [2]:
def gini(labels):
    n = len(labels)
    if n == 0:
        return 0
    counts = {}
    for l in labels:
        counts[l] = counts.get(l, 0) + 1
    return 1 - sum((c / n) ** 2 for c in counts.values())

def entropy(labels):
    n = len(labels)
    if n == 0:
        return 0
    counts = {}
    for l in labels:
        counts[l] = counts.get(l, 0) + 1
    return -sum((c / n) * math.log2(c / n) for c in counts.values())

X = [1, 2, 3, 4, 5, 6]
y = ["A", "A", "A", "B", "B", "B"]
print("parent  Gini=%.3f  Entropy=%.3f" % (gini(y), entropy(y)))
for t in (1, 3):
    L = [y[i] for i in range(6) if X[i] <= t]
    R = [y[i] for i in range(6) if X[i] > t]
    wG = (len(L)/6)*gini(L) + (len(R)/6)*gini(R)
    wH = (len(L)/6)*entropy(L) + (len(R)/6)*entropy(R)
    print(f"t={t}: L={L} R={R}")
    print(f"      weighted Gini={wG:.3f} -> Gini gain={gini(y)-wG:.3f} | "
          f"weighted Entropy={wH:.3f} -> Ent gain={entropy(y)-wH:.3f}")

parent  Gini=0.500  Entropy=1.000
t=1: L=['A'] R=['A', 'A', 'B', 'B', 'B']
      weighted Gini=0.400 -> Gini gain=0.100 | weighted Entropy=0.809 -> Ent gain=0.191
t=3: L=['A', 'A', 'A'] R=['B', 'B', 'B']
      weighted Gini=0.000 -> Gini gain=0.500 | weighted Entropy=-0.000 -> Ent gain=1.000


In [3]:
# "잘못 고를 확률" G=2p(1-p)와 "비트 수" H를 p의 함수로
p = np.linspace(0, 1, 401)
G = 2*p*(1-p)
with np.errstate(divide="ignore", invalid="ignore"):
    H = -(p*np.log2(p) + (1-p)*np.log2(1-p))
H[p==0] = H[p==1] = 0.0

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(p, G, lw=2.2, color="#1f77b4", label="Gini  $G=2p(1-p)$")
ax.plot(p, H, lw=2.2, color="#d62728", ls="--", label="Entropy  $H=-\sum p\log_2 p$ (bits)")
ax.axvline(0.5, color="gray", lw=0.8, ls=":")
ax.annotate("Half-half $p=0.5$\nGini 0.5 / H 1.0", xy=(0.5,0.5), xytext=(0.62,0.35),
            fontsize=9, arrowprops=dict(arrowstyle="->", lw=0.8))
ax.annotate("Pure $p=0,1$\nGini 0 / H 0", xy=(0.0,0.0), xytext=(0.08,-0.03),
            fontsize=9, arrowprops=dict(arrowstyle="->", lw=0.8))
ax.set_xlabel("Fraction of class 1  $p$")
ax.set_ylabel("Impurity (Gini) / Information (entropy, bits)")
ax.set_title("Gini impurity vs entropy — both maximal at $p=0.5$, zero when pure (0,1)")
ax.set_xlim(0,1); ax.set_ylim(-0.03,1.05)
ax.grid(alpha=0.3); ax.legend(fontsize=9, loc="center right")
fig.savefig(f"{IMG}/ch07_1_gini_entropy_curves.svg", bbox_inches="tight")
plt.show()
print("저장: ch07_1_gini_entropy_curves.svg")

저장: ch07_1_gini_entropy_curves.svg


## 2. 결정 트리 구조를 눈으로 (graphviz)

(1)의 6개 예시로 자란 트리를 graphviz로 그립니다. 루트는
"\(x \le 3\)?"을 쓰고, 두 리프는 순수한 클래스 A/B를 예측합니다 —
**내부 노드 = 질문, 리프 = 예측**이라는 구조를 한눈에 봅니다.
본문 상단 그림(`ch07_1_decision_tree_structure.svg`)과 동일합니다.

In [4]:
dot_src = '''
digraph decision_tree {
    rankdir=LR;
    graph [fontname="Noto Sans CJK KR", labelloc=t,
           label="Decision tree grown on 6 examples (x = 1-6, labels A,A,A,B,B,B) — internal nodes are questions, leaves are predictions",
           labeljust=c, bgcolor="white"];
    node  [fontname="Noto Sans CJK KR", shape=box, style="rounded,filled",
           fillcolor="white", fontcolor="#212529", fontsize=12.5,
           penwidth=1.4, margin="0.18,0.10"];
    edge  [fontname="Noto Sans CJK KR", fontsize=10.5, color="#495057", penwidth=1.6];

    R  [label="x ≤ 3 ?\nG = 0.5"];
    LA [label="Class A\n(3: x = 1, 2, 3)\nG = 0 (pure)", fillcolor="#cfe2ff"];
    LB [label="Class B\n(3: x = 4, 5, 6)\nG = 0 (pure)", fillcolor="#fff3cd"];

    R -> LA [label="Yes"];
    R -> LB [label="No"];
}
'''
g = graphviz.Source(dot_src)
g.format = "svg"
g.render(f"{IMG}/ch07_1_decision_tree_structure", cleanup=True)  # -> .svg
import os
p = f"{IMG}/ch07_1_decision_tree_structure.svg"
print("저장:", p, "(", os.path.getsize(p), "bytes )")

저장: /home/smhan/book-ml/kor/src/images/ch07_1_decision_tree_structure.svg ( 4345 bytes )


## 3. 실제 트리를 배우고 "읽기" (유방암 데이터)

30개 특징의 유방암 진단 데이터(양성/악성)를 7:3으로 나누고 **깊이 3**
트리를 학습시킵니다. 알고리즘이 **정보이득이 가장 큰 질문**(`worst
concave points`)을 루트에 놓는지 `export_text`로 확인하고, 예측 **경로**
(`decision_path`)와 **특징 중요도**(`feature_importances_`)를 봅니다.
"루트→리프 경로 = 설명"이라는 결정 트리의 핵심을 여기서 체감합니다.

In [5]:
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.3, random_state=0)

tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print(export_text(tree, feature_names=list(data.feature_names), max_depth=2))
print("train acc:", round(tree.score(X_train, y_train), 3))
print("test acc: ", round(tree.score(X_test, y_test), 3))


|--- worst concave points <= 0.14
|   |--- worst area <= 952.90
|   |   |--- area error <= 35.26
|   |   |   |--- class: 1
|   |   |--- area error >  35.26
|   |   |   |--- class: 1
|   |--- worst area >  952.90
|   |   |--- mean symmetry <= 0.15
|   |   |   |--- class: 1
|   |   |--- mean symmetry >  0.15
|   |   |   |--- class: 0
|--- worst concave points >  0.14
|   |--- area error <= 13.93
|   |   |--- class: 1
|   |--- area error >  13.93
|   |   |--- worst perimeter <= 79.13
|   |   |   |--- class: 1
|   |   |--- worst perimeter >  79.13
|   |   |   |--- class: 0

train acc: 0.967
test acc:  0.947


In [6]:
X_one = X_test[:1]
path = tree.decision_path(X_one)
print("경로를 지난 노드(노드번호):", path.indices)
print("예측 클래스:", tree.predict(X_one)[0], "(1=악성, 0=양성)")

imp = tree.feature_importances_
idx = np.argsort(imp)[::-1][:3]
print("\n상위 3 특징 중요도(지니 감소량 누적):")
for i in idx:
    print("  %-24s %.3f" % (data.feature_names[i], imp[i]))

경로를 지난 노드(노드번호): [ 0  8 10 12]
예측 클래스: 0 (1=악성, 0=양성)

상위 3 특징 중요도(지니 감소량 누적):
  worst concave points     0.817
  worst area               0.098
  area error               0.054


## 4. 깊이를 늘리면: U자 곡선 (과적합)

같은 유방암 데이터에서 `max_depth`를 1→12(그리고 `None`)로 늘리며
**train/test 정확도**를 재면, train은 1.0으로 오르는 동안 test는
정점(깊이 2~6, 0.947)을 지나 **0.912로 꺾인다** — 편향-분산
트레이드오프(U자)의 정석. "train이 1.0인 가장 깊은 트리"가
**가장 좋은 트리**가 아님을 보여줍니다.

In [7]:
depths = [1, 2, 3, 4, 5, 6, 8, 12, None]
tr_acc, te_acc, dlab = [], [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    tr_acc.append(m.score(X_train, y_train))
    te_acc.append(m.score(X_test, y_test))
    dlab.append(str(d))
    print(f"max_depth={str(d):>4}: train={tr_acc[-1]:.3f} test={te_acc[-1]:.3f} "
          f"leaves={m.get_n_leaves()}")

fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(range(len(depths)), tr_acc, "o-", lw=2, color="#1f77b4", label="train accuracy")
ax.plot(range(len(depths)), te_acc, "s-", lw=2, color="#d62728", label="test accuracy")
best = int(np.argmax(te_acc))
ax.axvline(best, color="gray", lw=1, ls="--")
ax.annotate(f"best test\n(depth={depths[best]}, {te_acc[best]:.3f})",
            xy=(best, te_acc[best]), xytext=(best+0.3, te_acc[best]+0.02),
            fontsize=9, arrowprops=dict(arrowstyle="->", lw=0.8))
ax.set_xticks(range(len(depths))); ax.set_xticklabels(dlab)
ax.set_xlabel("max_depth (1 → 12 → none)")
ax.set_ylabel("Accuracy")
ax.set_title("Deeper trees: train reaches 1.0, test bends into a U (overfitting)")
ax.set_ylim(0.85, 1.02); ax.grid(alpha=0.3); ax.legend(fontsize=9)
fig.savefig(f"{IMG}/ch07_1_depth_ucurve.svg", bbox_inches="tight")
plt.show()
print("저장: ch07_1_depth_ucurve.svg")

max_depth=   1: train=0.930 test=0.895 leaves=2
max_depth=   2: train=0.960 test=0.947 leaves=4
max_depth=   3: train=0.967 test=0.947 leaves=7


max_depth=   4: train=0.977 test=0.947 leaves=10


max_depth=   5: train=0.987 test=0.936 leaves=14
max_depth=   6: train=0.997 test=0.947 leaves=18
max_depth=   8: train=1.000 test=0.912 leaves=19
max_depth=  12: train=1.000 test=0.912 leaves=19
max_depth=None: train=1.000 test=0.912 leaves=19
저장: ch07_1_depth_ucurve.svg


## 5. 결정 경계는 왜 "계단"인가 (2D 장난감 데이터)

30개 특징은 2D로 그릴 수 없으니, 2개 특징의 작은 데이터(두 덩어리,
중심 \((0,0)\)·\((3,3)\), 오버랩 std=2.0, 400개 샘플)에서 **깊이 3**
트리와 **완전(깊이 제한 없음)** 트리의 결정 경계를 그립니다. 둘 다
**축에 평행한 직선 조각이 이어진 계단**이며, 완전 트리만 학습 데이터를
100% 맞히면서 경계가 점 사이로 파고들어 복잡해집니다 — 7.2절에서
랜덤 포레스트가 이 계단을 다수결로 "부드러운 곡선"으로 만드는 것과
대조됩니다.

In [8]:
X2, y2 = make_blobs(n_samples=400, cluster_std=2.0,
                    centers=[(0, 0), (3, 3)], random_state=0)
X2tr, X2te, y2tr, y2te = train_test_split(X2, y2, test_size=0.3, random_state=0)
t3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X2tr, y2tr)
tf = DecisionTreeClassifier(max_depth=None, random_state=0).fit(X2tr, y2tr)
print(f"깊이 3:  train={t3.score(X2tr,y2tr):.3f} test={t3.score(X2te,y2te):.3f} (리프 {t3.get_n_leaves()}개)")
print(f"완전:    train={tf.score(X2tr,y2tr):.3f} test={tf.score(X2te,y2te):.3f} (리프 {tf.get_n_leaves()}개)")

xx = np.linspace(X2[:,0].min()-0.5, X2[:,0].max()+0.5, 300)
yy = np.linspace(X2[:,1].min()-0.5, X2[:,1].max()+0.5, 300)
XX, YY = np.meshgrid(xx, yy)
grid = np.c_[XX.ravel(), YY.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(12,5))
for ax, clf, ttl in [(axes[0], t3, "Depth 3 (simple staircase)"),
                     (axes[1], tf, "Unpruned (digs between points = overfitting)")]:
    Z = clf.predict(grid).reshape(XX.shape)
    ax.contourf(XX, YY, Z, levels=[-0.5,0.5,1.5], colors=["#cfe2ff","#fff3cd"], alpha=0.8)
    ax.contour(XX, YY, Z, levels=[0.5], colors="k", lw=1.5)
    ax.scatter(X2tr[y2tr==0,0], X2tr[y2tr==0,1], s=18, c="#1f77b4", edgecolor="k", lw=0.4)
    ax.scatter(X2tr[y2tr==1,0], X2tr[y2tr==1,1], s=18, c="#d62728", edgecolor="k", lw=0.4)
    ax.set_title(ttl, fontsize=11)
    ax.set_xlabel("x1"); ax.set_ylabel("x2")
fig.suptitle("Decision-tree boundary — 'staircase' of axis-parallel straight-line segments", fontsize=12)
fig.tight_layout()
fig.savefig(f"{IMG}/ch07_1_boundary_staircase.svg", bbox_inches="tight")
plt.show()
print("저장: ch07_1_boundary_staircase.svg")

깊이 3:  train=0.846 test=0.792 (리프 8개)
완전:    train=1.000 test=0.733 (리프 59개)


저장: ch07_1_boundary_staircase.svg
